# SKQD on Fleets — deploy and run

This template runs **Sample-based Krylov Quantum Diagonalization (SKQD)** for a molecule,
built on [`qiskit-fermions`](https://github.com/Qiskit/qiskit-fermions), on IBM Code Engine
**Fleets** — across the CPU/GPU compute-profile ladder.

## What is SKQD?

SKQD estimates a molecule's ground-state energy by combining a quantum sampling step with a
classical diagonalization step:

1. Build the electronic-structure Hamiltonian.
2. Prepare a reference state and evolve it for increasing times — the **Krylov** family
   $\lvert\psi_k\rangle = e^{-i\,k\,\Delta t\,H}\lvert\psi_0\rangle$, $k = 0\dots D-1$.
3. **Sample** bitstrings from each of those $D$ circuits.
4. Project the Hamiltonian onto the subspace those configurations span and **diagonalize it
   classically** (`qiskit-addon-sqd`). The lowest eigenvalue is the SKQD energy estimate.

The estimate is a **variational upper bound**: it sits at or above the true ground-state
energy and improves as you sample a richer subspace.

## Stages

| Stage | Sub-status | What happens | Library |
|---|---|---|---|
| **Map** | `MAPPING` | PySCF SCF → MO integrals → `FermionOperator`; exact reference energy via a native FCI matvec. | pyscf / gpu4pyscf, qiskit-fermions |
| **Optimize** | `OPTIMIZING_HARDWARE` | Build the $D$ Krylov `FermionicCircuit`s and Jordan-Wigner-transpile them. | qiskit-fermions |
| **Execute** | `EXECUTING_QPU` | Sample each circuit (statevector, via ffsim). | ffsim |
| **Post-process** | `POST_PROCESSING` | The classical subspace eigensolve. | qiskit-addon-sqd |

The two heavy stages — building/sampling the Krylov circuits and the classical
diagonalization — both scale with the active space, which is what makes a large-memory / GPU
compute profile worthwhile on larger molecules. `qiskit-fermions` itself is CPU-only; the GPU
acceleration here is `gpu4pyscf` for the classical chemistry (the **Map** stage).

## 1. Authenticate and upload

This template uploads its `source_files/` directory to the gateway and runs on the standard
Fleets runtime image — no custom container required (the image ships `qiskit-fermions`, ffsim,
pyscf/gpu4pyscf, and qiskit-addon-sqd).

> **Requires** a Fleets-capable `qiskit-ibm-catalog` (the `runner="fleets"` / `sizes_map` /
> `function_size` API). Credentials are read from the environment.

The `sizes_map` maps t-shirt labels to Code Engine compute profiles: `s` is a CPU baseline;
`m`/`l`/`xl` add GPUs. The same uploaded code runs on any of them.

In [ ]:
import os
from qiskit_ibm_catalog import QiskitFunction, QiskitServerless

serverless = QiskitServerless(
    token=os.environ["QISKIT_IBM_TOKEN"],
    instance=os.environ["QISKIT_IBM_CRN"],
)

SIZES_MAP = {
    "s":  "24x120",          # 24 vCPU / 120 GB, CPU-only baseline
    "m":  "24x120x1l40s",    # + 1x L40s
    "l":  "20x320x2a100",    # + 2x A100
    "xl": "160x1792x8h100",  # + 8x H100
}

skqd_fleets = QiskitFunction(
    title="skqd_fleets",
    entrypoint="skqd_entrypoint.py",
    working_dir="./source_files/",
    runner="fleets",
    sizes_map=SIZES_MAP,
    default_size="s",
)
serverless.upload(skqd_fleets)

In [ ]:
function = serverless.load("skqd_fleets")
print("function:", function.title, "| sizes:", function.sizes())

## 2. Run

Every argument is optional; defaults run a linear H₄ chain in STO-3G. Pick a `function_size` to
choose the compute profile (`"m"` for a GPU, `"s"` for the CPU baseline). See the README for a
full table of the tunable knobs and what each one does.

In [ ]:
job = function.run(
    function_size="m",          # compute profile from sizes_map; "s" = CPU baseline
    # --- system (defaults = linear H4 / STO-3G) ---
    # atom=[["H", [0.0, 0.0, 1.0 * i]] for i in range(4)],
    # basis="sto-3g",
    # --- convergence / cost ---
    krylov_dim=5,
    shots=2000,
    samples_per_batch=100,
    num_batches=3,
    max_iterations=3,
)
print(job.job_id, job.status())

In [ ]:
result = job.result()  # blocks until done; raises with the error message on failure
print(f"SKQD energy:      {result['energy']:.6f} Ha")
print(f"exact reference:  {result['reference_energy']:.6f} Ha")
print(f"error:            {result['error'] * 1e3:.2f} mHa")
print(f"ran on GPU:       {result['ran_on_gpu']}")
print(f"configurations:   {result['num_configurations']}")
print(f"timings (s):      {result['metadata']['timings']}")

## 3. Interpret

- **`energy`** — the SKQD estimate; **`reference_energy`** — exact diagonalization for comparison.
- **`error`** — `|energy - reference|`. SKQD is variational, so `energy >= reference`. Chemical
  accuracy is ~1.6 mHa; the small default settings will be looser — tighten the convergence
  knobs (README) to close the gap.
- **`ran_on_gpu` / `gpu_check`** — whether a GPU was attached and the stack loaded on it.
- **`metadata.timings`** — per-stage wall-clock. On small systems **map** dominates; on large
  ones **diagonalize** takes over — that crossover is the reason for the compute-profile ladder.

## 4. Sweep a knob (optional)

The defaults are deliberately small. Sweep the Krylov dimension to watch the energy converge
toward the exact reference.

In [ ]:
krylov_dims = [1, 3, 5, 8, 12]
jobs = {
    d: function.run(function_size="m", krylov_dim=d, shots=4000, num_batches=5, max_iterations=5)
    for d in krylov_dims
}
results = {d: j.result() for d, j in jobs.items()}
reference = next(iter(results.values()))["reference_energy"]
for d in krylov_dims:
    r = results[d]
    print(f"krylov_dim={d:2d}  energy={r['energy']:.6f}  error={r['error'] * 1e3:6.2f} mHa")
print(f"exact reference = {reference:.6f} Ha")

In [ ]:
import matplotlib.pyplot as plt

energies = [results[d]["energy"] for d in krylov_dims]
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(krylov_dims, energies, marker="o", label="SKQD energy")
ax.axhline(reference, color="gray", linestyle="--", label="exact reference")
ax.set_xlabel("Krylov dimension $D$")
ax.set_ylabel("energy (Ha)")
ax.set_title("SKQD energy vs Krylov dimension")
ax.legend()
fig